In [1]:
from nlp4bia.datasets.benchmark.symptemist import SymptemistLoader, SymptemistGazetteer

In [4]:
from nlp4bia.datasets.Dataset import BenchmarkDataset
from nlp4bia.datasets import config
from nlp4bia.datasets.utils import handlers
import os
        
from requests import get
from zipfile import ZipFile
from io import BytesIO
import pandas as pd

            
class SymptemistLoader(BenchmarkDataset):
    URL = "https://zenodo.org/records/10635215/files/symptemist-complete_240208.zip?download=1"
    NAME = "symptemist-complete_240208"
    DS_COLUMNS = config.DS_COLUMNS
    
    def __init__(self, lang="es", path=None, name=NAME, url=URL, download_if_missing=True):
        super().__init__(lang, name, path, url, download_if_missing)

    def load_data(self):
        '''Load the data from the dataset
        Output: DataFrame with columns: filename, mark, label, off0, off1, span, code, semantic_rel, split, text
        '''
        
        train_path = os.path.join(self.path, "symptemist_train/subtask2-linking/symptemist_tsv_train_subtask2_complete+COMPOSITE.tsv")
        texts_train_path = os.path.join(self.path, "symptemist_train/subtask1-ner/txt")
        test_path = os.path.join(self.path, "symptemist_test/subtask2-linking/symptemist_tsv_test_subtask2+COMPOSITE.tsv")
        texts_test_path = os.path.join(self.path, "symptemist_test/subtask1-ner/txt")

        df_train = pd.read_csv(train_path, sep="\t", dtype=str)
        df_test = pd.read_csv(test_path, sep="\t", dtype=str)
        
        df_train["split"] = "train"
        df_test["split"] = "test"
        
        df = pd.concat([df_train, df_test])
        df.rename(columns={"text": "span"}, inplace=True)
        
        df_texts = handlers.get_texts(texts_train_path, texts_test_path)
        df = df.merge(df_texts, on="filename", how="left")
        
        self.df = df
        
        return df
    
    def preprocess_data(self):
        print("preprocessing data...")
        # DS_COLUMNS =  ["filenameid", "mention_class", "span", "code", "sem_rel", "is_abbreviation", "is_composite", "needs_context", "extension_esp"]
        
        d_map_names = {"label": "mention_class", "need_context": "needs_context"}
        
        self.df["filenameid"] = self.df["filename"] + "#" + self.df["span_ini"] + "#" + self.df["span_end"]
        self.df.drop(columns=["filename", "span_ini", "span_end"], inplace=True)

        self.df.rename(columns=d_map_names, inplace=True)
        
        for col in self.DS_COLUMNS:
            if col not in self.df.columns:
                self.df[col] = None
        
        cols = self.DS_COLUMNS + ["text", "split"]
        self.df = self.df[cols]
        
        assert self.df.columns.intersection(self.DS_COLUMNS).shape[0] == len(self.DS_COLUMNS), "There are missing columns"
        
        return self.df
        
    def _download_data(self, download_path):
        # Ensure download path exists
        os.makedirs(download_path, exist_ok=True)

        # Download dataset
        print("Downloading dataset...")
        temp_zip_path = os.path.join(download_path, "temp_dataset.zip")
        handlers.progress_download(self.URL, temp_zip_path)

        # Extract if zip file
        with ZipFile(temp_zip_path, 'r') as zip_file:
            zip_file.extractall(download_path)
        
        # Clean up the temporary zip file
        os.remove(temp_zip_path)
        print("Dataset downloaded and extracted successfully.")

        return download_path

In [5]:
sl = SymptemistLoader()
sl.df

preprocessing data...


,filenameid,mention_class,span,code,sem_rel,is_abbreviation,is_composite,needs_context,extension_esp,text,split
0,es-S0365-66912011000600005-2#333#361,SINTOMA,«manchas» en el campo visual,246658005,EXACT,None,False,False,None,Varón de 37 años ex-adicto a drogas por vía pa...,train
1,es-S0004-06142010000300011-1#649#716,SINTOMA,5HIAA en orina de 24 horas estaba dentro de lo...,171250001,NARROW,None,False,False,None,Paciente de 24 años con un hermano gemelo que ...,train
2,es-S1130-01082007000700011-2#1463#1505,SINTOMA,A nivel analítico no presentaba alteración,166315009,NARROW,None,False,False,None,"Mujer de 68 años, como antecedentes personales...",train
3,es-S0210-48062009000300017-1#2713#2759,SINTOMA,a nivel del cardias masa mamelonada y ulcerada,126825008,NARROW,None,False,False,None,Paciente de 57 años de edad remitido a urgenci...,train
4,es-S1130-01082006000100014-1#2282#2295,SINTOMA,abdomen agudo,9209005,EXACT,None,False,False,None,"Se trata de una mujer de 35 años, con antecede...",train
...,...,...,...,...,...,...,...,...,...,...,...
12009,es-S1130-01082007001100009-1#393#446,SINTOMA,zona indurada en la pared lateral izquierda de...,5964004,NARROW,None,False,False,None,Mujer de 42 años estudiada en Consultas de Gas...,test
12010,es-S0212-71992005000600008-1#903#997,SINTOMA,"zona periumbilical, donde se aprecia a la insp...",448569009,NARROW,None,False,False,None,"Varón de 71 años, que ingresó en el servicio d...",test
12011,es-S1139-76322016000300008-1#1642#1699,SINTOMA,zona superior de la vejiga hacia el ombligo un...,235993005,NARROW,None,False,False,None,Presentamos el caso de una lactante de cinco m...,test
12012,es-S1134-80462005000300004-1#2808#2869,SINTOMA,zonas hipoestésicas y espásticas en ambos miem...,NO_CODE,NO_CODE,None,False,False,None,Presentamos el caso de un varón de 53 años de ...,test


In [ ]:
path = sl.path
path

'/home/abecerr1/.nlp4bia/symptemist-complete_240208'

In [ ]:
train_path = "symptemist_train/subtask2-linking/"
df_cc = pd.read_csv(os.path.join(path, train_path, "symptemist_tsv_train_subtask2_complete+COMPOSITE.tsv"), sep="\t")
df_c = pd.read_csv(os.path.join(path, train_path, "symptemist_tsv_train_subtask2_complete.tsv"), sep="\t")
df = pd.read_csv(os.path.join(path, train_path, "symptemist_tsv_train_subtask2.tsv"), sep="\t")
print(df_cc.shape)
print(df_c.shape)
print(df.shape)
print("Training documents:", df_cc.filename.unique().shape, df_c.filename.unique().shape, df.filename.unique().shape)
print(np.intersect1d(df_cc.filename.unique(), df.filename.unique()).shape == df.filename.unique().shape)
print(np.intersect1d(df_c.filename.unique(), df.filename.unique()).shape == df.filename.unique().shape)
print(np.intersect1d(df_cc.filename.unique(), df_c.filename.unique()).shape == df_c.filename.unique().shape)

(8980, 10)
(8276, 10)
(3484, 10)
Training documents: (744,) (744,) (304,)
True
True
True


In [ ]:
test_path = "symptemist_test/subtask2-linking/"
df_t = pd.read_csv(os.path.join(path, test_path, "symptemist_tsv_test_subtask2+COMPOSITE.tsv"), sep="\t")
df_t.shape

(3034, 10)

In [ ]:
df_cc

,filename,label,span_ini,span_end,text,code,sem_rel,is_composite,is_abbrev,need_context
0,es-S0365-66912011000600005-2,SINTOMA,333,361,«manchas» en el campo visual,246658005,EXACT,False,False,False
1,es-S0004-06142010000300011-1,SINTOMA,649,716,5HIAA en orina de 24 horas estaba dentro de lo...,171250001,NARROW,False,False,False
2,es-S1130-01082007000700011-2,SINTOMA,1463,1505,A nivel analítico no presentaba alteración,166315009,NARROW,False,False,False
3,es-S0210-48062009000300017-1,SINTOMA,2713,2759,a nivel del cardias masa mamelonada y ulcerada,126825008,NARROW,False,False,False
4,es-S1130-01082006000100014-1,SINTOMA,2282,2295,abdomen agudo,9209005,EXACT,False,False,False
...,...,...,...,...,...,...,...,...,...,...
8975,es-S1134-80462009000800005-1,SINTOMA,745,750,dolor,22253000,EXACT,False,False,False
8976,es-S1134-80462009000800005-1,SINTOMA,2587,2598,somnolencia,271782001,EXACT,False,False,False
8977,es-S1134-80462009000800005-1,SINTOMA,2136,2143,vómitos,422400008,EXACT,False,False,False
8978,es-S1134-80462009000800005-1,SINTOMA,2988,3006,sensación nauseosa,422587007,EXACT,False,False,False


In [ ]:
from nlp4bia.datasets import config
import requests
from tqdm import tqdm

download("https://zenodo.org/records/10635215/files/symptemist-complete_240208.zip?download=1", os.path.join(config.NLP4BIA_DATA_PATH, "symptemist-complete_240208.zip"))

NameError: name 'download' is not defined